# Resource Selection Function

In [ ]:
import pandas as pd
import ee

from movement_models import (
    FeatureSpec,
    _build_design_matrix,
    _get_availability_domain,
    _get_sampling_points,
    _sample_env_layer,
    load_presence_csvs,
    to_reloc_gdf,
    reproject_reloc,
    to_reloc_gdf_projected,
    init_ee,
    init_ee_on_client,
    make_aoi,
    aoi_to_ee,
    build_predictors_image,
    ee_image_to_env_xarray,
    save_env_zarr,
    load_env_zarr,
    shutdown_default_client,
    make_local_dask_client,
    
    fit_rsf,
    predict_rsf_points,
    get_rsf_surface,
    fixed_width_Boyce,
    sliding_window_Boyce,
    cv_model,
    eval_all_linear_candidates
)

In [ ]:
client = make_local_dask_client(memory_limit="2GB") 

init_status = init_ee_on_client(client, project_id="ee-alvykabo")
client, init_status

## 1. Preparation

### 1.1 Fetch presence data

In [ ]:
reloc_df = load_presence_csvs("data/*.csv")
reloc = to_reloc_gdf(reloc_df)

reloc.head(), reloc.crs, reloc["Timestamp"].dtype

### 1.2 Fetch environmental layers

In [ ]:
ee.Authenticate()
init_ee(project_id="ee-alvykabo")   

aoi = make_aoi(reloc, buffer_m=1)
aoi_ee = aoi_to_ee(aoi)

predictors_img = build_predictors_image(aoi_ee, start="2024-01-01", end="2024-12-31")
env = ee_image_to_env_xarray(predictors_img, aoi_ee, crs="EPSG:29333", scale=100, chunk_xy=1024)

save_env_zarr(env, "env_29333.zarr", mode="w")
env2 = load_env_zarr("env_29333.zarr")

env.shape, env.rio.crs, env2.shape

## 2. Calculate the RSF

### 2.1 Define the sampling scheme

In [ ]:
domain = _get_availability_domain(reloc)
samples = _get_sampling_points(domain, 1000, df=reloc.iloc[:100], seed=1)
sampled = _sample_env_layer(samples, env.sel(band=["ndvi", "slope"]), chunk_size_points="auto", client=client)

sampled.head()

### 2.2 Extract values from the environmental covariates

### 2.3 Fit logistic regression

In [ ]:
spec = FeatureSpec(linear=["ndvi", "slope"], add_const=True)
m, scaler, _ = fit_rsf(sampled, spec)
pred = predict_rsf_points(sampled, m, scaler, spec)

pred[["rsf_pred"]].describe()

### 2.4 Get RSF surface

In [ ]:
rsf = get_rsf_surface(env.sel(band=spec.linear), m, scaler, spec)

rsf

In [ ]:
rsf = get_rsf_surface(env, m, scaler, spec)
rsf = rsf.compute()
rsf.rio.to_raster("rsf.tif", compress="LZW")
rsf = rsf.rio.write_crs("EPSG:29333") 

### 2.5 Evaluate Using Boyce Index

In [ ]:
subset_predictors = ["ndvi", "ndwi", "slope", "dist2water"]
cov = env.sel(band=subset_predictors)

samples = _get_sampling_points(domain, n=10_000, df=reloc.iloc[:1000], seed=1)
sampled = _sample_env_layer(samples, cov, chunk_size_points="auto", client=client)

spec = FeatureSpec(linear=subset_predictors, add_const=True)
m, scaler, spec = fit_rsf(sampled, spec)

pred = predict_rsf_points(sampled, m, scaler, spec)
rsf = get_rsf_surface(env.sel(band=subset_predictors), m, scaler, spec)

B, boyce = fixed_width_Boyce(pred, rsf, domain, seed=1)
B, boyce.head()

In [ ]:
B, chart = sliding_window_Boyce(
    pred=pred,
    rsf=rsf,
    domain=domain,
    window_frac=0.2,
    step_frac=0.05,
    seed=1,
)

print("B =", B)
print(chart.head())
print("rows:", len(chart), "finite pe:", chart["pe"].notna().sum())

## 3. Aggregate functions

### 3.1 Compare all possible linear models via AIC & BIC

In [ ]:
df_small = sampled.sample(n=min(2000, len(sampled)), random_state=1).copy()
subset_predictors = ["ndvi", "ndwi", "slope", "dist2water"]

df_small = df_small.replace([float("inf"), float("-inf")], pd.NA).dropna(subset=["used"] + subset_predictors)

res = eval_all_linear_candidates(df_small, env, subset=subset_predictors)
res.head()

### 3.2 Cross-Validation on a Single Individual

In [ ]:
res = cv_model(
    obs=reloc,
    env=env,
    k_folds=2,
    subset_predictors=["ndvi", "ndwi", "slope", "dist2water"],
    sampling_factor_train=3,
    n_bg_boyce=5_000,
    seed=1,
)
res